# ADM Pro-Forma Valuation Workbench

Interactive FIN 43900 Project 1 Step 4 workbench. Amounts are USD millions except percentages and per-share values. Every output is calculated by the unmodified Step 2 integrated model; the controls only change its assumptions.

## Colab setup

In a fresh Google Colab session, run this cell. If the required model files are not already in the Colab working directory, upload **both** `adm_integrated_model.py` and `adm_valuation_model.py` from this project folder. The first is your preserved Step 2 model; the second is a thin adapter that calls it.

In [ ]:
# Colab includes pandas and ipywidgets; this makes a fresh runtime explicit.
!pip -q install ipywidgets pandas
from pathlib import Path
from google.colab import files

required_files = ('adm_integrated_model.py', 'adm_valuation_model.py')
missing_files = [filename for filename in required_files if not Path(filename).is_file()]
if missing_files:
    print(f"Upload the missing files: {', '.join(missing_files)}")
    files.upload()
    missing_files = [filename for filename in required_files if not Path(filename).is_file()]
assert not missing_files, f"Missing required model files: {', '.join(missing_files)}"

Upload the missing files: adm_integrated_model.py, adm_valuation_model.py


Saving adm_integrated_model.py to adm_integrated_model.py
Saving adm_valuation_model.py to adm_valuation_model.py


In [ ]:
from copy import deepcopy
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from adm_valuation_model import base_assumptions, run_workbench, failure_test_assumptions

BASE = base_assumptions()  # exact, unrounded Step 2 assumptions
BASE_RESULT = run_workbench(BASE)
YEARS = list(BASE_RESULT['income_statement'])
print(f"Loaded Step 2 base case. WACC: {BASE_RESULT['dcf']['wacc']:.6%}; value/share: ${BASE_RESULT['per_share_valuation']:.6f}")

Loaded Step 2 base case. WACC: 6.689404%; value/share: $-0.740876


## Controls

The selected operating assumptions apply uniformly to FY2026E–FY2030E, consistent with the Step 2 base case. Ranges are deliberately broad but economically bounded: growth −5% to 10%; gross margin 3% to 15%; SG&A 40% to 85% of gross profit; inventory 30–80 days; CapEx $500m–$2,500m; WACC 5%–12%; and terminal growth 0%–4.5%. The 4.5% maximum terminal growth remains below the 5.0% minimum WACC, preventing an invalid Gordon-growth denominator.

Use **Reset to Base** to restore exact Step 2 values. **Run documented failure test** loads the existing Step 2 Low-style stress (0% growth, 6.25% gross margin, 53 inventory days, $1,600m CapEx, 2.0% terminal growth, and the Step 2 Low-case WACC); it should exceed the $6,000m facility limit in 2030.

In [ ]:
def percent_control(label, key, minimum, maximum, step=0.0005):
    return widgets.FloatSlider(value=BASE[key], min=minimum, max=maximum, step=step,
                               description=label, readout_format='.2%',
                               style={'description_width': 'initial'}, layout=widgets.Layout(width='430px'))

growth = percent_control('Revenue growth', 'revenue_growth', -0.05, 0.10)
margin = percent_control('Gross margin', 'gross_margin', 0.03, 0.15)
sga = percent_control('SG&A / gross profit', 'sga_pct_gross_profit', 0.40, 0.85)
inventory = widgets.FloatSlider(value=BASE['inventory_days'], min=30, max=80, step=1, description='Inventory days',
                                  readout_format='.0f', style={'description_width': 'initial'}, layout=widgets.Layout(width='430px'))
capex = widgets.FloatSlider(value=BASE['capex'], min=500, max=2500, step=50, description='Annual CapEx ($m)',
                           readout_format='.0f', style={'description_width': 'initial'}, layout=widgets.Layout(width='430px'))
wacc = percent_control('WACC', 'wacc', 0.05, 0.12, 0.0001)
terminal = percent_control('Terminal growth', 'terminal_growth', 0.00, 0.045)
controls = [growth, margin, sga, inventory, capex, wacc, terminal]
reset = widgets.Button(description='Reset to Base', button_style='primary', icon='refresh')
failure_test = widgets.Button(description='Run documented failure test', button_style='danger', icon='warning', layout=widgets.Layout(width='250px'))
out = widgets.Output()

def selected_assumptions():
    return {'revenue_growth': growth.value, 'gross_margin': margin.value,
            'sga_pct_gross_profit': sga.value, 'inventory_days': inventory.value,
            'capex': capex.value, 'wacc': wacc.value, 'terminal_growth': terminal.value}

def statement_frame(statement):
    return pd.DataFrame(statement).rename_axis('Line item').round(1)

def validation_frame(result):
    rows = []
    for label, detail in result['validation_results'].items():
        row = {'Check': label, 'Status': 'PASS' if detail['pass'] else 'FAIL'}
        row.update({str(year): value for year, value in detail['values'].items()})
        rows.append(row)
    return pd.DataFrame(rows).set_index('Check').round(2)

def render(*_):
    assumptions = selected_assumptions()
    with out:
        clear_output(wait=True)
        try:
            result = run_workbench(assumptions)
        except ValueError as error:
            display(HTML(f"<h3 style='color:#b00020'>INVALID INPUT: {error}</h3>"))
            return
        dcf = result['dcf']
        difference = result['per_share_valuation'] - BASE_RESULT['per_share_valuation']
        display(HTML(f"<h2>DCF summary</h2><table><tr><th align='left'>WACC</th><td>{dcf['wacc']:.2%}</td></tr><tr><th align='left'>Enterprise value</th><td>${dcf['enterprise_value']:,.1f}m</td></tr><tr><th align='left'>Equity value</th><td>${dcf['equity_value']:,.1f}m</td></tr><tr><th align='left'>Implied value/share</th><td>${result['per_share_valuation']:,.2f}</td></tr><tr><th align='left'>Difference from base/share</th><td>${difference:,.2f}</td></tr></table>"))
        if result['facility_failures']:
            year, balance = next(iter(result['facility_failures'].items()))
            excess = balance - result['facility_limit']
            display(HTML(f"<div style='padding:14px;border:3px solid #b00020;background:#ffe8e8;color:#760014'><b>FAIL — FINANCIALLY INFEASIBLE</b><br>Forecast year: {year}E<br>Facility balance: ${balance:,.1f}m<br>Facility limit: ${result['facility_limit']:,.1f}m<br>Excess borrowing: ${excess:,.1f}m<br>The current financing assumptions cannot fund this forecast; borrowing is shown, not clipped.</div>"))
        else:
            display(HTML('<p style="color:#157347"><b>PASS:</b> liquidity facility remains within its $6,000m limit.</p>'))
        display(HTML('<h3>Annual operating income and FCFF</h3>'))
        display(pd.DataFrame({'Operating income (EBIT)': result['annual_operating_income'], 'FCFF': result['fcff']}).T.round(1))
        display(HTML('<h3>Liquidity facility balance</h3>'))
        display(pd.DataFrame([result['liquidity_facility_balances']], index=['Facility balance']).round(1))
        display(HTML('<h3>Five-year income statement</h3>')); display(statement_frame(result['income_statement']))
        display(HTML('<h3>Five-year balance sheet</h3>')); display(statement_frame(result['balance_sheet']))
        display(HTML('<h3>Five-year cash flow statement</h3>')); display(statement_frame(result['cash_flow_statement']))
        display(HTML('<h3>Validation checks</h3>')); display(validation_frame(result))

def set_controls(values):
    for control in controls:
        control.unobserve(render, names='value')
    try:
        for control, key in zip(controls, ['revenue_growth', 'gross_margin', 'sga_pct_gross_profit', 'inventory_days', 'capex', 'wacc', 'terminal_growth']):
            control.value = values[key]
    finally:
        for control in controls:
            control.observe(render, names='value')
    render()

def reset_to_base(_): set_controls(BASE)
def run_failure_test(_): set_controls(failure_test_assumptions())
reset.on_click(reset_to_base)
failure_test.on_click(run_failure_test)
for control in controls: control.observe(render, names='value')
display(widgets.VBox([widgets.HTML('<h3>Adjustable assumptions</h3>'), *controls, widgets.HBox([reset, failure_test], layout=widgets.Layout(align_items='center')), out]))
render()

## Validation checklist

1. Immediately after loading, the base case should show approximately 6.69% WACC, $7,299.9m enterprise value, −$357.1m equity value, −$0.74/share, and a 2030 facility balance of $5,144.2m.
2. Move a control and confirm the linked statements, FCFF, DCF, and facility balance change.
3. Select **Reset to Base** and confirm the original values return.
4. Select **Run documented failure test** and confirm the red warning reports the 2030 $6,873.6m facility balance, $6,000m limit, and $873.6m excess (displayed amounts are rounded only for presentation).

The validation table checks balance-sheet balance, cash reconciliation, retained earnings, debt roll-forward, the minimum cash policy, and the liquidity-facility limit in every forecast year.